# C4 SE supervised simulation (`se-supervised-sim-v0`)

Synthetic prototypes:
- **A.** Next-attempt correctness
- **B.** Future mock-exam score (+ optional pass if classes support it)

Explicitly synthetic. **Not** real SE / real-exam validation. Does **not** retrain ASSISTments or SE MaskablePPO.

Run cells **in order**. Full training and final-test are gated (`False` by default).


## 1. Dependencies


In [ ]:
%pip install -q "numpy>=1.23,<3" "pandas>=1.5,<3" "scikit-learn>=1.2,<2" "scipy>=1.10,<2" "joblib>=1.2,<2" matplotlib tqdm


## 2. Upload & extract `c4-se-supervised-source-colab.zip`


In [ ]:
from pathlib import Path
import zipfile, shutil
from google.colab import files

UPLOAD_DIR = Path('/content/c4_se_sup_upload')
EXTRACT_DIR = Path('/content/c4_se_sup_src')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print('Upload c4-se-supervised-source-colab.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded')
zip_name = next(iter(uploaded))
zip_path = UPLOAD_DIR / zip_name
zip_path.write_bytes(uploaded[zip_name])
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)
print('Top:', list(EXTRACT_DIR.iterdir())[:10])


Upload c4-se-supervised-source-colab.zip


Saving c4-se-supervised-source-colab.zip to c4-se-supervised-source-colab.zip
Extracted to /content/c4_se_sup_src
Top: [PosixPath('/content/c4_se_sup_src/ai-services')]


## 3. Drive mount


In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts root:', DRIVE_ROOT)


Artifacts root: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0


## 4. Imports / versions / paths


In [ ]:
import sys, os, json
from pathlib import Path

if 'EXTRACT_DIR' not in globals() or 'DRIVE_ROOT' not in globals():
    raise RuntimeError('Run cells 2 (extract) and 3 (Drive mount) first.')

hits = list(Path(EXTRACT_DIR).rglob('training_se_supervised/protocol.py'))
if not hits:
    raise FileNotFoundError('training_se_supervised/protocol.py not found; re-upload ZIP')
C4_ROOT = hits[0].parents[1].resolve()
sys.path.insert(0, str(C4_ROOT))
os.chdir(C4_ROOT)
print('C4_ROOT', C4_ROOT)

from training_se_supervised.colab_bootstrap import require_symbol
from training_se_supervised.protocol import PROTOCOL_VERSION, GENERATOR_VERSION, DOMAIN_MARKERS
from training_se_supervised.provenance import capture_versions
print('protocol', PROTOCOL_VERSION, GENERATOR_VERSION)
print('markers', DOMAIN_MARKERS)
print(json.dumps(capture_versions(), indent=2))


C4_ROOT /content/c4_se_sup_src/ai-services/c4
protocol se-supervised-sim-v0 se_supervised_gen_v0
markers {'domain': 'se_simulation', 'synthetic': True, 'simulation_only': True, 'real_exam_validation': False, 'protocol': 'se-supervised-sim-v0', 'generator_version': 'se_supervised_gen_v0'}
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "protocol": "se-supervised-sim-v0",
  "generator_version": "se_supervised_gen_v0",
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "joblib": "1.6.0",
  "scipy": "1.16.3"
}


**bold text**## 5. Frozen generation / evaluation contract + data inventory


In [ ]:
from training_se_supervised.contract import frozen_contract_dict, PASS_THRESHOLD, SCENARIOS
from training_se_supervised.inventory import inventory_local_se_exports
from training_se_supervised.catalogue import catalogue_hash, ORDERED_TOPICS

CONTRACT = frozen_contract_dict(mode='smoke', master_seed=42)
print('pass_threshold', PASS_THRESHOLD)
print('topics', ORDERED_TOPICS)
print('catalogue_sha256', catalogue_hash())
print('scenarios', list(SCENARIOS))
print(json.dumps(CONTRACT, indent=2)[:2500])
print(json.dumps(inventory_local_se_exports(C4_ROOT), indent=2))

# Persist a copy under Drive for the session audit trail
(DRIVE_ROOT / '_session_contract_preview.json').write_text(
    json.dumps(CONTRACT, indent=2), encoding='utf-8'
)


pass_threshold 0.55
topics ('usability_heuristics', 'user_research_methods', 'prototyping', 'normalization', 'indexing', 'query_optimization', 'case_analysis_techniques', 'industry_case_studies', 'lessons_learned', 'risk_management', 'estimation', 'agile_planning')
catalogue_sha256 5001dd005443e61a27118d094ee02bd3c39fb668bbc7222dc95951e3f0624543
scenarios ['baseline', 'high_noise', 'easy_bank']
{
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v0",
  "generator_version": "se_supervised_gen_v0",
  "contract": {
    "protocol": "se-supervised-sim-v0",
    "generator_version": "se_supervised_gen_v0",
    "master_seed": 42,
    "pass_threshold": 0.55,
    "pass_threshold_note": "Pass = (exam items correct / exam_n_items) >= 0.55. Threshold declared a priori for the synthetic mock exam; not an institutional cutoff.",
    "pass_min_class_count": 8,
    "primary_scenario": "baseline",
    "split": {

6092

## 6. Smoke generation + leakage checks + train/evaluate on validation


In [ ]:
from training_se_supervised.pipeline import run_training_and_validation

smoke = run_training_and_validation(
    DRIVE_ROOT,
    mode='smoke',
    master_seed=42,
    c4_root=C4_ROOT,
)
SMOKE_RUN_DIR = Path(smoke['run_dir'])
print('smoke run', SMOKE_RUN_DIR)
print('selection', json.dumps(smoke['selection'], indent=2))
print('leakage', smoke['leakage'])


smoke run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/train_smoke_se-supervised-sim-v0_20261012T092837.258Z
selection {
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v0",
  "generator_version": "se_supervised_gen_v0",
  "primary_scenario": "baseline",
  "response": {
    "name": "logistic_regression",
    "val_metrics": {
      "n": 194,
      "prevalence": 0.36597938144329895,
      "pred_mean": 0.3858025220510937,
      "log_loss": 0.6535471390356175,
      "brier": 0.22832067822395483,
      "roc_auc": 0.6288789648459865
    },
    "primary_metric": "log_loss"
  },
  "exam_score": {
    "name": "hist_gradient_boosting_regressor",
    "val_metrics": {
      "n": 9,
      "mae": 0.1354277605109785,
      "rmse": 0.17878914563729573,
      "y_mean": 0.3703703703703704,
      "y_std": 0.19257534199808402,
      "pred_mean": 0.44477137254138405,
      "pred_std": 0.152

## 7. Validation selection table


In [ ]:
import pandas as pd

require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
sel = json.loads((SMOKE_RUN_DIR / 'selection_val.json').read_text())
resp = json.loads((SMOKE_RUN_DIR / 'comparison_val_response.json').read_text())
exam = json.loads((SMOKE_RUN_DIR / 'comparison_val_exam_score.json').read_text())

df_resp = pd.DataFrame([
    {
        'target': 'response',
        'model': r['name'],
        'val_log_loss': r['val_metrics'].get('log_loss'),
        'val_brier': r['val_metrics'].get('brier'),
        'val_roc_auc': r['val_metrics'].get('roc_auc'),
        'prevalence': r['val_metrics'].get('prevalence'),
        'selected': r['name'] == sel['response']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
    }
    for r in resp
])
df_exam = pd.DataFrame([
    {
        'target': 'exam_score',
        'model': r['name'],
        'val_mae': r['val_metrics'].get('mae'),
        'val_rmse': r['val_metrics'].get('rmse'),
        'y_mean': r['val_metrics'].get('y_mean'),
        'selected': r['name'] == sel['exam_score']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
    }
    for r in exam
])
display(df_resp)
display(df_exam)
print('All scenarios reported in scenario_sensitivity.json (not used to pick generator).')


,target,model,val_log_loss,val_brier,val_roc_auc,prevalence,selected,synthetic,domain
0,response,constant_train_prevalence,0.672135,0.239522,0.500000,0.365979,False,True,se_simulation
1,response,logistic_regression,0.653547,0.228321,0.628879,0.365979,True,True,se_simulation
2,response,hist_gradient_boosting_classifier,0.684845,0.238618,0.614107,0.365979,False,True,se_simulation


,target,model,val_mae,val_rmse,y_mean,selected,synthetic,domain
0,exam_score,train_mean,0.198743,0.218022,0.37037,False,True,se_simulation
1,exam_score,ridge_regression,0.216979,0.263435,0.37037,False,True,se_simulation
2,exam_score,hist_gradient_boosting_regressor,0.135428,0.178789,0.37037,True,True,se_simulation


All scenarios reported in scenario_sensitivity.json (not used to pick generator).


## 8. Full data generation + training (gated)


In [11]:
# Set True only for the intended full synthetic run. Creates a NEW timestamped folder.
RUN_FULL_TRAINING = True

FULL_RUN_DIR = None
if RUN_FULL_TRAINING:
    full = run_training_and_validation(
        DRIVE_ROOT,
        mode='full',
        master_seed=42,
        run_full=True,
        c4_root=C4_ROOT,
    )
    FULL_RUN_DIR = Path(full['run_dir'])
    print('full run', FULL_RUN_DIR)
    print('selection', json.dumps(full['selection'], indent=2))
else:
    print('Skipped full training (RUN_FULL_TRAINING=False)')


full run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/train_full_se-supervised-sim-v0_20261012T130156.573Z
selection {
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v0",
  "generator_version": "se_supervised_gen_v0",
  "primary_scenario": "baseline",
  "response": {
    "name": "hist_gradient_boosting_classifier",
    "val_metrics": {
      "n": 13748,
      "prevalence": 0.5040733197556008,
      "pred_mean": 0.5084976621937548,
      "log_loss": 0.6145887086372498,
      "brier": 0.21327592486465474,
      "roc_auc": 0.7198541696561643
    },
    "primary_metric": "log_loss"
  },
  "exam_score": {
    "name": "ridge_regression",
    "val_metrics": {
      "n": 300,
      "mae": 0.08241833727718875,
      "rmse": 0.10316841432793439,
      "y_mean": 0.5411111111111111,
      "y_std": 0.17595275144759584,
      "pred_mean": 0.5336668180160974,
      "pred_std": 0.1591

In [12]:
from pathlib import Path
import json
import pandas as pd

assert FULL_RUN_DIR is not None, "Run Section 8 first."
RUN_DIR = Path(FULL_RUN_DIR)

selection = json.loads(
    (RUN_DIR / "selection_val.json").read_text()
)

for target, filename in [
    ("response", "comparison_val_response.json"),
    ("exam_score", "comparison_val_exam_score.json"),
    ("pass", "comparison_val_pass.json"),
]:
    comparisons = json.loads((RUN_DIR / filename).read_text())
    chosen = selection[target].get("name")

    print(f"\nTARGET: {target} | Selected: {chosen}")
    if comparisons:
        display(pd.DataFrame([
            {
                "model": row["name"],
                **row["val_metrics"],
                "selected": row["name"] == chosen,
            }
            for row in comparisons
        ]))
    else:
        print(selection[target])

print("\nLeakage checks:")
print(json.loads((RUN_DIR / "leakage_checks.json").read_text()))
print("\nFull run folder:", RUN_DIR)


TARGET: response | Selected: hist_gradient_boosting_classifier


,model,brier,log_loss,n,pred_mean,prevalence,roc_auc,selected
0,constant_train_prevalence,0.250288,0.693724,13748,0.521527,0.504073,0.500000,False
1,logistic_regression,0.214455,0.617530,13748,0.509450,0.504073,0.716795,False
2,hist_gradient_boosting_classifier,0.213276,0.614589,13748,0.508498,0.504073,0.719854,True



TARGET: exam_score | Selected: ridge_regression


,model,mae,n,pred_mean,pred_std,rmse,y_mean,y_std,selected
0,train_mean,0.145705,300,0.550159,1.112078e-16,0.175892,0.541111,0.175953,False
1,ridge_regression,0.082418,300,0.533667,1.591931e-01,0.103168,0.541111,0.175953,True
2,hist_gradient_boosting_regressor,0.083401,300,0.531213,1.608302e-01,0.104886,0.541111,0.175953,False



TARGET: pass | Selected: logistic_regression


,model,brier,log_loss,n,pred_mean,prevalence,roc_auc,selected
0,constant_train_prevalence,0.251570,0.696295,300,0.536429,0.496667,0.500000,False
1,logistic_regression,0.142164,0.431191,300,0.488213,0.496667,0.884350,True
2,hist_gradient_boosting_classifier,0.150050,0.455444,300,0.489252,0.496667,0.878795,False



Leakage checks:
{'checks': ['learner_disjoint', 'chronology', 'exam_after_practice', 'response_pre_attempt_features', 'exam_pre_exam_features', 'no_audit_latents_in_features', 'no_identifiers_as_features', 'target_ranges'], 'exam_score_mean': 0.5506388888888888, 'n_exam_rows': 2000, 'n_response_rows': 92767, 'ok': True, 'pass_prevalence': 0.5365, 'response_prevalence': 0.5175331745124883}

Full run folder: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/train_full_se-supervised-sim-v0_20261012T130156.573Z


In [13]:
from pathlib import Path
import json

from training_se_supervised.contract import frozen_contract_dict
from training_se_supervised.pipeline import run_final_evaluation

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_supervised_sim_v0/"
    "train_full_se-supervised-sim-v0_20261012T130156.573Z"
)

assert RUN_DIR.is_dir(), f"Missing run folder: {RUN_DIR}"

saved = json.loads(
    (RUN_DIR / "frozen_contract.json").read_text()
)
contract = saved["contract"]

assert contract["mode"] == "full"

current = frozen_contract_dict(
    mode=contract["mode"],
    master_seed=contract["master_seed"],
)

if saved != current:
    raise RuntimeError(
        "Saved contract differs from current code settings. "
        "Stop here; do not evaluate with changed settings."
    )

selection = json.loads(
    (RUN_DIR / "selection_val.json").read_text()
)
frozen = json.loads(
    (RUN_DIR / "SELECTION_FROZEN.json").read_text()
)
assert selection == frozen, "Validation selection has changed."

for filename in (
    "response_selected.joblib",
    "exam_score_selected.joblib",
    "pass_selected.joblib",
):
    assert (RUN_DIR / "models" / filename).is_file(), filename

if (RUN_DIR / "final_test").exists():
    raise RuntimeError(
        "Final-test folder already exists. "
        "Inspect existing results instead of overwriting."
    )

print("Saved contract matches current settings.")
print("Frozen selections:")
for target in ("response", "exam_score", "pass"):
    print(target, "->", frozen[target]["name"])

final = run_final_evaluation(RUN_DIR, run_final=True)

print("\nFinal evaluation completed. No retraining.")
print("Results:", RUN_DIR / "final_test/final_test_metrics.json")
print("Export:", RUN_DIR / "export_bundle")

Saved contract matches current settings.
Frozen selections:
response -> hist_gradient_boosting_classifier
exam_score -> ridge_regression
pass -> logistic_regression

Final evaluation completed. No retraining.
Results: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/train_full_se-supervised-sim-v0_20261012T130156.573Z/final_test/final_test_metrics.json
Export: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/train_full_se-supervised-sim-v0_20261012T130156.573Z/export_bundle


In [14]:
import pandas as pd

rows = []
for target, key in (
    ("response", "response_test"),
    ("exam_score", "exam_score_test"),
    ("pass", "pass_test"),
):
    metrics = final.get(key)
    if metrics is not None:
        rows.append({
            "target": target,
            "selected_model": frozen[target]["name"],
            **metrics,
        })

display(pd.DataFrame(rows))

print("Synthetic test learners only; real_exam_validation=False.")

,target,selected_model,n,prevalence,pred_mean,log_loss,brier,roc_auc,mae,rmse,y_mean,y_std,pred_std
0,response,hist_gradient_boosting_classifier,14194,0.512329,0.513649,0.610846,0.211496,0.725332,NaN,NaN,NaN,NaN,NaN
1,exam_score,ridge_regression,300,NaN,0.551901,NaN,NaN,NaN,0.076438,0.096883,0.562407,0.181105,0.166221
2,pass,logistic_regression,300,0.576667,0.538366,0.370711,0.118238,0.915844,NaN,NaN,NaN,NaN,NaN


Synthetic test learners only; real_exam_validation=False.


In [15]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import zipfile

import training_se_supervised
from training_se_supervised.model_cards import (
    write_response_model_card,
    write_exam_score_model_card,
    write_pass_model_card,
)
from training_se_supervised.export_bundle import (
    export_run_bundle,
    build_manifest,
    verify_manifest,
)

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_supervised_sim_v0/"
    "train_full_se-supervised-sim-v0_20261012T130156.573Z"
)

def read_json(name):
    return json.loads((RUN_DIR / name).read_text())

selection = read_json("SELECTION_FROZEN.json")
final = read_json("final_test/final_test_metrics.json")
cards = RUN_DIR / "model_cards"
cards.mkdir(exist_ok=True)

# Update cards from saved results only.
write_response_model_card(
    cards / "MODEL_CARD_RESPONSE.md",
    {
        "selection": selection["response"],
        "comparison_val": read_json("comparison_val_response.json"),
        "primary_metric": "log_loss",
        "primary_scenario": selection["primary_scenario"],
        "final_test": final["response_test"],
    },
)

write_exam_score_model_card(
    cards / "MODEL_CARD_EXAM_SCORE.md",
    {
        "selection": selection["exam_score"],
        "comparison_val": read_json("comparison_val_exam_score.json"),
        "final_test": final["exam_score_test"],
    },
)

write_pass_model_card(
    cards / "MODEL_CARD_PASS.md",
    {
        "selection": selection["pass"],
        "comparison_val": read_json("comparison_val_pass.json"),
    },
)

# Existing pass-card writer does not include final metrics.
with (cards / "MODEL_CARD_PASS.md").open("a", encoding="utf-8") as f:
    f.write(
        "\n## Final synthetic test\n\n```json\n"
        + json.dumps(final["pass_test"], indent=2)
        + "\n```\n"
    )

for card in cards.glob("*.md"):
    with card.open("a", encoding="utf-8") as f:
        f.write(
            "\nScenario comparisons refit models within each scenario; "
            "they are not transfer evaluations of one frozen model.\n"
        )

BUNDLE = export_run_bundle(RUN_DIR)

# Include source needed to load custom joblib wrapper classes.
source_root = Path(training_se_supervised.__file__).resolve().parents[1]

for package in ("training_se_supervised", "training_se"):
    source = source_root / package
    assert source.is_dir(), f"Missing source package: {source}"
    for file in source.rglob("*.py"):
        if "__pycache__" in file.parts:
            continue
        destination = (
            BUNDLE / "source" / package / file.relative_to(source)
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, destination)

(BUNDLE / "SOURCE_README.txt").write_text(
    "Source snapshot from the loaded Colab package at export time.\n"
    "Custom joblib classes require these packages on Python's import path.\n"
    "Do not overwrite live application source automatically.\n"
    "See runtime_versions.json for the recorded training environment.\n"
    "All models are synthetic simulation prototypes.\n",
    encoding="utf-8",
)

for name in (
    "response_selected.joblib",
    "exam_score_selected.joblib",
    "pass_selected.joblib",
):
    assert (BUNDLE / "models" / name).is_file(), name

build_manifest(BUNDLE)
verification = verify_manifest(BUNDLE)
assert verification["ok"], verification
print("Checksums:", verification)

exports = RUN_DIR.parent / "exports"
exports.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ZIP_PATH = exports / f"c4_se_supervised_bundle_{stamp}.zip"

with zipfile.ZipFile(
    ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file in sorted(BUNDLE.rglob("*")):
        if file.is_file():
            archive.write(
                file,
                "export_bundle/" + file.relative_to(BUNDLE).as_posix(),
            )

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None, "ZIP integrity failed."

print("Package:", ZIP_PATH)
print("Size MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))

Checksums: {'ok': True, 'n': 61, 'mismatches': []}
Package: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v0/exports/c4_se_supervised_bundle_20261012T131046728530Z.zip
Size MB: 0.15


In [16]:
from google.colab import files
files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from training_se_supervised.pipeline import _prepare_tables
from training_se_supervised.models_response import (
    load_response_bundle,
    predict_response_bundle,
)
from training_se_supervised.models_exam import (
    load_exam_bundle,
    predict_pass_bundle,
)

RUN = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_supervised_sim_v0/"
    "train_full_se-supervised-sim-v0_20261012T130156.573Z"
)

contract = json.loads(
    (RUN / "frozen_contract.json").read_text()
)["contract"]

# Feature builder uses this threshold; confirm it matches the saved run.
from training_se_supervised.features_exam import PASS_THRESHOLD
assert PASS_THRESHOLD == contract["pass_threshold"]

split = json.loads((RUN / "split_ids.json").read_text())
id_sets = {
    key: set(values)
    for key, values in split["learner_ids"].items()
}

data = RUN / "synthetic_data/scenarios" / contract["primary_scenario"]
practice = pd.read_csv(data / "practice_attempts.csv")
exams = pd.read_csv(data / "mock_exams.csv")

response_rows, exam_rows = _prepare_tables(
    practice,
    exams,
    max_response_rows=contract["sizes"]["max_response_rows_per_learner"],
    id_sets=id_sets,
)

response_test = response_rows[response_rows["split"] == "test"]
exam_test = exam_rows[exam_rows["split"] == "test"]

response_model = load_response_bundle(
    RUN / "models/response_selected.joblib"
)
pass_model = load_exam_bundle(
    RUN / "models/pass_selected.joblib"
)

# Fixed probability decision threshold; no threshold tuning on test data.
DECISION_THRESHOLD = 0.50

results = []
for target, model, frame, label, predict in [
    (
        "Next-answer correctness",
        response_model,
        response_test,
        "y_correct",
        predict_response_bundle,
    ),
    (
        "Synthetic mock-exam pass",
        pass_model,
        exam_test,
        "y_pass",
        predict_pass_bundle,
    ),
]:
    probabilities = predict(model, frame)
    predicted = (probabilities >= DECISION_THRESHOLD).astype(int)
    actual = frame[label].to_numpy(dtype=int)
    correct = int(np.sum(predicted == actual))

    results.append({
        "target": target,
        "model": model["name"],
        "test_rows": len(actual),
        "correct_predictions": correct,
        "accuracy_percent": round(100 * correct / len(actual), 2),
        "probability_threshold": DECISION_THRESHOLD,
        "data": "synthetic",
    })

display(pd.DataFrame(results))

,target,model,test_rows,correct_predictions,accuracy_percent,probability_threshold,data
0,Next-answer correctness,hist_gradient_boosting_classifier,14194,9425,66.40,0.5,synthetic
1,Synthetic mock-exam pass,logistic_regression,300,248,82.67,0.5,synthetic


## 9. Final-test evaluation (gated; loads frozen selection — no config fallback)


In [ ]:
from training_se_supervised.pipeline import run_final_evaluation, reload_run

RUN_FINAL_EVALUATION = False
# Point at a run that already has SELECTION_FROZEN.json + frozen_contract.json
FINAL_TARGET_RUN = FULL_RUN_DIR or globals().get('SMOKE_RUN_DIR')

if RUN_FINAL_EVALUATION:
    require_symbol('FINAL_TARGET_RUN', FINAL_TARGET_RUN)
    # Demonstrate reload after restart
    reloaded = reload_run(FINAL_TARGET_RUN)
    print('reloaded selection keys', list(reloaded['selection'].keys()))
    final = run_final_evaluation(FINAL_TARGET_RUN, run_final=True)
    print(json.dumps(final, indent=2))
else:
    print('Skipped final evaluation (RUN_FINAL_EVALUATION=False)')


## 10. Export reminder


In [ ]:
from training_se_supervised.export_bundle import verify_manifest

target = globals().get('FULL_RUN_DIR') or globals().get('SMOKE_RUN_DIR')
if target is None:
    print('No run directory in this session')
else:
    export_dir = Path(target) / 'export_bundle'
    print('export_bundle', export_dir)
    print(verify_manifest(export_dir))
    print('Download export_bundle from Drive. Marks: domain=se_simulation, synthetic=true,')
    print('simulation_only=true, real_exam_validation=false.')
    print('Do NOT auto-wire into se-path-sim-v0 MaskablePPO dynamics.')
